In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report, accuracy_score, precision_score, recall_score
from torch.utils.data import DataLoader, Subset
from transformers import get_cosine_schedule_with_warmup
from huggingface_hub import snapshot_download
from transformers import Wav2Vec2Config, Wav2Vec2Model
from tqdm.notebook import tqdm

class ArcFaceLoss(nn.Module):
    def __init__(self, num_classes=4, embedding_size=256, margins=None, margin=None, scale=20.0, class_weights=None, gamma=2.0, **kwargs):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.weight)

        if margins is None:
            margins = margin if margin is not None else [0.45, 0.20, 0.40, 0.45]

        self.scale = nn.Parameter(torch.tensor(scale, dtype=torch.float32))
        self.register_buffer('margins', torch.tensor(margins, dtype=torch.float32))
        self.register_buffer("class_weights", class_weights if class_weights is not None else torch.ones(num_classes))
        self.gamma = gamma

    def forward(self, embeddings, labels):
        normalized_embeddings = F.normalize(embeddings, p=2, dim=1)
        normalized_weights = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(normalized_embeddings, normalized_weights)

        cosine = torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7)

        target_margins = self.margins[labels]
        cos_m = torch.cos(target_margins)
        sin_m = torch.sin(target_margins)

        sine_inner = torch.clamp(1.0 - torch.pow(cosine, 2), min=1e-7)
        sine = torch.sqrt(sine_inner)
        phi = cosine * cos_m.unsqueeze(1) - sine * sin_m.unsqueeze(1)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)

        current_scale = torch.clamp(self.scale, min=10.0, max=40.0)
        output *= current_scale

        ce_loss = F.cross_entropy(output, labels, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.class_weights[labels] * ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class SERModel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        config = Wav2Vec2Config.from_pretrained(MODEL_NAME)
        config.output_hidden_states = True
        config.mask_time_prob = 0.4
        config.mask_feature_prob = 0.15
        config.mask_time_length = 10

        self.wav2vec2 = Wav2Vec2Model(config)
        self.wav2vec2.gradient_checkpointing_disable()
        self.wav2vec2.freeze_feature_encoder()

        save_dir = snapshot_download(MODEL_NAME, allow_patterns=["*.ckpt", "*.safetensors", "*.bin"])
        ckpt_files = glob.glob(os.path.join(save_dir, "**/*.ckpt"), recursive=True) + \
                     glob.glob(os.path.join(save_dir, "**/*.bin"), recursive=True) + \
                     glob.glob(os.path.join(save_dir, "**/*.safetensors"), recursive=True)

        if ckpt_files:
            try:
                ckpt_file = ckpt_files[0]
                if ckpt_file.endswith('.safetensors'):
                    from safetensors.torch import load_file
                    raw_state = load_file(ckpt_file)
                else:
                    raw_state = torch.load(ckpt_file, map_location="cpu", weights_only=False)

                state_dict = raw_state.get("state_dict", raw_state.get("model", raw_state)) if isinstance(raw_state, dict) else raw_state
                prefixes = ["mods.wav2vec2.model.", "mods.wav2vec2.", "model.wav2vec2.model.", "model.wav2vec2.", "wav2vec2.", "model.", "net.", "modules."]
                clean_dict = {}
                for k, v in state_dict.items():
                    nk = k
                    for p in prefixes:
                        if nk.startswith(p):
                            nk = nk[len(p):]
                            break
                    clean_dict[nk] = v

                model_state = self.wav2vec2.state_dict()
                final_dict = {}
                for k, v in clean_dict.items():
                    if k in model_state and model_state[k].shape == v.shape:
                        final_dict[k] = v
                    else:
                        for mk, mv in model_state.items():
                            if (mk.endswith(k) or k.endswith(mk)) and mv.shape == v.shape:
                                final_dict[mk] = v
                                break
                self.wav2vec2.load_state_dict(final_dict, strict=False)
                print(f"[HỆ THỐNG] Nạp thành công {len(final_dict)} tensor từ Custom Backbone.")
            except Exception as e:
                print(f"[CẢNH BÁO CRITICAL] Lỗi nạp trọng số tùy chỉnh: {e}")

        for i in range(4):
            for param in self.wav2vec2.encoder.layers[i].parameters():
                param.requires_grad = False

        hidden_size = config.hidden_size
        self.layer_weights = nn.Parameter(torch.ones(13))
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.feat_drop = nn.Dropout(0.3)

        self.bigru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.att_proj = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.sap_norm = nn.LayerNorm(hidden_size * 2)
        self.fc1 = nn.Linear(hidden_size * 2, 256)
        self.gelu = nn.GELU()

        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.2, 0.3, 0.4, 0.5, 0.6]])

    def forward(self, input_values, attention_mask=None):
        outputs = self.wav2vec2(
            input_values=input_values,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )

        stacked_states = torch.stack(outputs.hidden_states, dim=0)
        num_states = stacked_states.shape[0]

        if self.layer_weights.shape[0] != num_states:
            if self.layer_weights.shape[0] > num_states:
                active_weights = self.layer_weights[:num_states]
            else:
                active_weights = F.pad(self.layer_weights, (0, num_states - self.layer_weights.shape[0]))
        else:
            active_weights = self.layer_weights

        norm_weights = F.softmax(active_weights, dim=0)
        weighted_feat = torch.sum(norm_weights.view(-1, 1, 1, 1) * stacked_states, dim=0)
        weighted_feat = self.feat_drop(self.layer_norm(weighted_feat))

        if attention_mask is not None:
            input_lens = attention_mask.sum(dim=-1)
            feature_lens = self.wav2vec2._get_feat_extract_output_lengths(input_lens)
            batch_size, seq_len, _ = weighted_feat.shape
            mask = torch.arange(seq_len, device=weighted_feat.device).expand(batch_size, seq_len) < feature_lens.unsqueeze(1)
            weighted_feat = weighted_feat * mask.unsqueeze(-1).float()
        else:
            mask = None

        gru_out, _ = self.bigru(weighted_feat)
        attn_scores = self.att_proj(gru_out).squeeze(-1)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(~mask, -1e4)

        attn_weights = F.softmax(attn_scores, dim=-1).unsqueeze(-1)
        weighted_mean = torch.sum(attn_weights * gru_out, dim=1)
        diff = gru_out - weighted_mean.unsqueeze(1)
        weighted_var = torch.sum(attn_weights * (diff ** 2), dim=1)
        weighted_std = torch.sqrt(weighted_var + 1e-8)

        sap_out = torch.cat([weighted_mean, weighted_std], dim=-1)
        x = self.sap_norm(sap_out)
        x = self.gelu(self.fc1(x))

        embeddings = torch.mean(torch.stack([dropout(x) for dropout in self.dropouts], dim=0), dim=0)
        return embeddings

def plot_confusion_matrix(y_true, y_pred, fold_idx, f1_score_val):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=list(LABEL2ID.keys()), yticklabels=list(LABEL2ID.keys()))
    plt.title(f'Fold {fold_idx} Confusion Matrix (Macro F1: {f1_score_val:.4f})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    plt.close()